# N109 · 树形DP与局部约束

**目标：** 用节点选与不选状态表达相邻冲突。

**先修：** N071, N098, N101。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 后序；子树独立；状态合并；树背包/匹配扩展；选取方案恢复。

## 从问题到算法

树上没有跨子树边，因此给定父节点状态后，各子树可以独立优化。选点问题保存“选当前点/不选当前点”。摄像头把子树压成三种状态：当前未覆盖、当前放相机、已被子节点覆盖；空孩子视为已覆盖，避免在空位置安装相机。

## 最小实现

**本章接口：** `rob_tree(root)`、`min_camera_cover(root)`、`tree_independent_set(adj, weights)`

In [1]:
class TreeNode:
    def __init__(self,val=0,left=None,right=None):
        self.val,self.left,self.right=val,left,right

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None: return None
    root=TreeNode(values[0]); queue=deque([root]); i=1
    while queue and i<len(values):
        node=queue.popleft()
        if values[i] is not None: node.left=TreeNode(values[i]); queue.append(node.left)
        i+=1
        if i<len(values):
            if values[i] is not None: node.right=TreeNode(values[i]); queue.append(node.right)
            i+=1
    return root
def _postorder_nodes(root):
    stack=[(root,False)]
    while stack:
        node,done=stack.pop()
        if not node: continue
        if done: yield node
        else:
            stack.append((node,True)); stack.append((node.right,False)); stack.append((node.left,False))
def _tree_order(adj,root=0):
    if not adj: return [],[]
    parent=[-1]*len(adj); parent[root]=root; order=[root]
    for u in order:
        for v in adj[u]:
            if v==parent[u]: continue
            if parent[v]!=-1: raise ValueError('tree required')
            parent[v]=u; order.append(v)
    if len(order)!=len(adj): raise ValueError('connected tree required')
    return parent,order

def rob_tree(root):
    dp={None:(0,0)}
    for node in _postorder_nodes(root):
        l0,l1=dp[node.left]; r0,r1=dp[node.right]
        dp[node]=(max(l0,l1)+max(r0,r1),node.val+l0+r0)
    return max(dp[root])

def min_camera_cover(root):
    state={None:2}; count=0
    for node in _postorder_nodes(root):
        left,right=state[node.left],state[node.right]
        if 0 in (left,right): state[node]=1; count+=1
        elif 1 in (left,right): state[node]=2
        else: state[node]=0
    return count+(root is not None and state[root]==0)

def tree_independent_set(adj,weights):
    if len(adj)!=len(weights): raise ValueError('one weight per vertex')
    parent,order=_tree_order(adj); dp=[[0,0] for _ in adj]
    for u in reversed(order):
        dp[u][1]=weights[u]
        for v in adj[u]:
            if parent[v]==u: dp[u][0]+=max(dp[v]); dp[u][1]+=dp[v][0]
    return max(dp[0]) if dp else 0

## 正确性、前提与复杂度

独立集转移直接来自父子不能同时选。摄像头若某孩子未覆盖，放在当前点可同时覆盖该孩子、当前点及父点，不劣于在孩子上补装；孩子均已覆盖且无相机时，把是否覆盖当前点的决定留给父点最灵活。根没有父点，最后必须单独补相机。

**代价：** O(n)时间，本实现用字典保存所有子树状态，辅助O(n)，不将其误写为仅O(h)。邻接表接口要求连通无向树；二叉树对象不得含共享孩子或环。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

root=tree_from_level([3,2,3,None,3,None,1]); dp={None:(0,0)}; rows=[]
for node in _postorder_nodes(root):
    a,b=dp[node.left]; c,d=dp[node.right]; dp[node]=(max(a,b)+max(c,d),node.val+a+c); rows.append((node.val,*dp[node]))
show_table(['节点值','不选','选'],rows)
print('最少相机',min_camera_cover(root))

节点值,不选,选
3,0,3
2,3,2
1,0,1
3,1,3
3,6,7


最少相机 2


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert rob_tree(tree_from_level([3,2,3,None,3,None,1]))==7
assert min_camera_cover(TreeNode())==1 and min_camera_cover(None)==0
from random import Random
rng=Random(109)
for n in range(1,9):
    for _ in range(20):
        nodes=[TreeNode(rng.randrange(-2,7)) for _ in range(n)]; edges=[]
        available=[0]
        for v in range(1,n):
            p=rng.choice(available); node=nodes[p]
            if node.left is None: node.left=nodes[v]
            else: node.right=nodes[v]; available.remove(p)
            available.append(v); edges.append((p,v))
        adj=[[] for _ in range(n)]
        for u,v in edges: adj[u].append(v); adj[v].append(u)
        best=max(sum(nodes[i].val for i in range(n) if mask>>i&1) for mask in range(1<<n) if all(not(mask>>u&1 and mask>>v&1) for u,v in edges))
        assert rob_tree(nodes[0])==tree_independent_set(adj,[x.val for x in nodes])==best
        cameras=min(mask.bit_count() for mask in range(1<<n) if all(mask>>i&1 or any(mask>>j&1 for j in adj[i]) for i in range(n)))
        assert min_camera_cover(nodes[0])==cameras
print('N109: 所有本章断言通过')

N109: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 把二叉树推广到一般树。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 构造只用一个最优值丢失父子兼容信息的例子。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 337. House Robber III（canonical）
- 968. Binary Tree Cameras（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。